# 🧮 第 08 课 · KV Cache 的内存账本:一个 token 占多少显存?

> KV 占用公式推导 · MHA / GQA / MQA 三兄弟 · 真实 CUDA 分配对照 · vLLM 块预算

**章节**: 第 2 章 · KV Cache 与 PagedAttention

---

> **📚 本笔记本属于《minivllm: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 从注意力张量形状出发,严格推导每 token 的 KV 内存公式 bytes = 2×L×H_kv×D×dtype_bytes
- 用形状为 (2, B, L, H_kv, S, D) 的手工张量逐行推演,核对 numel × 字节 与公式一致
- 理解 MHA / GQA / MQA 的数学关系:GQA 把 H_q 个 query 头分 G 组共享 KV 头,缓存缩小 G 倍
- 在真实 GPU 上分配与公式同形状的 CUDA 张量,用 memory_allocated 验证理论 = 实测
- 代入 LLaMA-3 8B / 70B、Qwen2.5 等真实模型参数,算出单请求与高并发下的显存需求
- 复现 vLLM 启动时的块预算逻辑:num_gpu_blocks = 剩余显存 ÷ 每块字节数

## 📑 本课目录

1. **直觉:记在显存账本上的每一笔** — 为什么 KV Cache 常常比模型权重还大
2. **核心公式与符号表** — 每个符号的含义与维度,一步推到底
3. **手工张量逐行推演** — 构造 (2,B,L,H_kv,S,D),numel × 字节 = 公式
4. **MHA / GQA / MQA 三兄弟** — GQA 论文 (arXiv:2305.13245) 的共享头数学
5. **真实规模数字** — LLaMA-3 / Qwen / Mistral 配置表的真实账本
6. **真实 GPU 分配验证** — real_ops.kv_bytes_real:理论 vs 实测偏差 < 0.01%
7. **fp8 精度旋钮** — dtype_bytes 2→1 让账本直接减半
8. **与 vLLM 的关系** — gpu_memory_utilization 与 num_gpu_blocks 怎么定
9. **Streamlit 动态演示** — 交互式 KV 内存计算器

## 🔗 参考资料

- [PagedAttention 论文 (SOSP'23)](https://arxiv.org/abs/2309.06180)
- [GQA: Multi-Query Transformer (arXiv)](https://arxiv.org/abs/2305.13245)
- [MQA: One Write-Head is All You Need (arXiv)](https://arxiv.org/abs/1911.02150)
- [NVIDIA: LLM Inference Optimization](https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/)
- [vLLM 官方文档](https://docs.vllm.ai)

## 1. 直觉:记在显存账本上的每一笔

上一课我们证明了 KV Cache 把生成的计算量从 $O(T^2)$ 降到 $O(T)$。但**省下来的 K/V 不能凭空消失**——
它们被**存在显存(HBM)里**,而且每个 token 都要为**每一层、每个 KV 头**各存一份 K 和一份 V。

把显存想象成一本**大账本**:

- 一个 token 的 K 记一笔、V 再记一笔(**因子 2**);
- 模型有 $L$ 层,每层的注意力各算各的,都要单独记(**因子 L**);
- 每层有 $H_{kv}$ 个 KV 头(GQA 里是共享的 KV 头),每个头都要记(**因子 H_{kv}**);
- 每个头是一个 $D$ 维向量,每个元素占 `dtype_bytes` 字节(**因子 D × dtype_bytes**)。

于是一个 token 的「账」是:

$$ \text{Bytes / token} = 2 \times L \times H_{kv} \times D \times \text{dtype\_bytes} $$

推理时**所有并发请求的所有 token 一起记账**。请求越多、对话越长,账本越厚——
在实际部署中,**KV Cache 往往超过模型权重,成为最大的显存开销**。NVIDIA 官方博客明确指出 LLM 显存的两大主体就是「模型权重 + KV Cache」([Inference Optimization](https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/))。

> 🏷️ 本课目标:把这个公式推导清楚,并用**真实 GPU 分配**验证它分毫不差。

## 2. 核心公式与符号表:每个符号都要搞清楚

严格写法(把 batch 与序列长度也显式写出,这就是全库通用的 KV 内存公式):

$$ \text{KV bytes} = 2 \times \underbrace{B}_{\text{batch}} \times S \times L \times H_{kv} \times D \times \text{dtype\_bytes} $$

| 符号 | 含义 | 取值范围/默认 |
|---|---|---|
| $2$ | K 与 V 各一份 | 恒定 |
| $B$ | 并发请求数 batch | 1 ~ 数百 |
| $S$ | 序列长度 seq_len(每请求 token 数) | 1K ~ 128K |
| $L$ | Transformer 层数 | 28 ~ 126 |
| $H_{kv}$ | **KV 头数**(MHA 时 = 注意力头数 $H$;GQA 时 = 共享组数;MQA 恒为 1) | 1 ~ 96 |
| $D$ | 每头维度 head_dim | 64 ~ 256,主流 128 |
| `dtype_bytes` | 每元素字节:fp32=4,bf16/fp16=2,fp8=1 | 1/2/4 |

> ⚠️ 两个易错点:
> 1. 公式里用的是 **$H_{kv}$(KV 头数)而不是 $H$(query 头数)**——GQA 模型里两者相差 G 倍。    用错会导致账本虚高;读模型 `config.json` 时看 `num_key_value_heads`。
> 2. **没有 `d_model`**:当且仅当 MHA 时 $H_{kv}\times D = d_{model}$ 才成立;    GQA/MQA 下二者不相等。

工程上还常把「每 token 成本」单独拎出来(不乘 $B,S$),它是显存规划的**最小单元**,后面算 vLLM 块预算全靠它。

## 3. 手工张量逐行推演:把公式兑现成真实张量

先取一组**可以心算**的迷你参数,构造一个与真实 KV cache 同构的张量:

```
batch B=2    (2 个并发请求)
layers L=3   (3 层 Transformer)
kv_heads H_kv=4  (GQA 的 4 个共享 KV 头)
head_dim D=8 (每头 8 维)
seq S=16     (每请求 16 个 token)
dtype=bf16   (每元素 2 字节)
```

张量形状取 `(2, B, L, H_kv, S, D)`,第 0 维的 **2 = K 与 V 两层**——这正是 vLLM/aios 的布局。

✅ **逐行推演**。关键结论:`numel × dtype_bytes` 与公式 `2×B×S×L×H_kv×D×dtype` 严格相等,而「每 token 成本」与 B、S 无关——它是后面所有容量计算的最小单元。

In [ ]:
# -*- coding: utf-8 -*-
import torch

# --------------------------------------------------------------------------
# 1. 定义迷你模型的超参数(刻意取小值,让每个数字都能心算验证)
# --------------------------------------------------------------------------
batch_size = 2          # B: 并发请求数(同时处理 2 条对话)
num_layers = 3          # L: Transformer 层数(每层一套独立的 KV 缓存)
kv_heads = 4            # H_kv: KV 头数(GQA 中多个 query 头共享的 KV 头)
head_dim = 8            # D: 每个注意力头的向量维度(头向量有多长)
seq_len = 16            # S: 单请求序列长度(已缓存多少 token)
dtype_bytes = 2         # bf16/fp16 每元素 2 字节

# --------------------------------------------------------------------------
# 2. 构造与真实 KV cache 同构的张量,形状 (2, B, L, H_kv, S, D)
#    第 0 维的 2 = K 与 V 两层(K 一层、V 一层,正是公式里开头的因子 2)
# --------------------------------------------------------------------------
kv_cache = torch.zeros(
    2, batch_size, num_layers, kv_heads, seq_len, head_dim,   # 六个维度的尺寸
    dtype=torch.bfloat16,                                     # bf16 每元素 2 字节
)
print(f"KV cache 张量 shape = {tuple(kv_cache.shape)}")
print("  维度含义: (K/V=2, batch=B, layers=L, kv_heads=H_kv, seq=S, head_dim=D)")

# --------------------------------------------------------------------------
# 3. 用「元素个数 × 每元素字节」独立算一遍字节数
# --------------------------------------------------------------------------
n_elements = kv_cache.numel()                                 # 张量里的元素总数
theory = n_elements * dtype_bytes                             # 元素数 × 2 字节
print(f"\n元素总数 numel = {n_elements:,}")
print(f"理论字节 = {n_elements:,} × {dtype_bytes} = {theory:,} bytes = {theory / 2**20:.3f} MiB")

# --------------------------------------------------------------------------
# 4. 用 KV 内存公式独立算一遍,并核对两者相等
# --------------------------------------------------------------------------
formula = (2 * batch_size * seq_len * num_layers * kv_heads
           * head_dim * dtype_bytes)                          # 公式:2×B×S×L×H_kv×D×dtype
print(f"公式字节 = 2×{batch_size}×{seq_len}×{num_layers}×{kv_heads}×{head_dim}×{dtype_bytes} = {formula:,}")
print(f"numel×字节 与 公式 一致? {theory == formula}  ✅")

# --------------------------------------------------------------------------
# 5. 提炼「每 token 成本」:总字节 ÷ token 总数
# --------------------------------------------------------------------------
n_tokens = batch_size * seq_len                               # 全部请求的 token 总数
per_token = theory / n_tokens                                 # 每个 token 摊到的字节
print(f"\n全部 token 数 = {n_tokens},每 token = {per_token:.0f} bytes = {per_token/1024:.1f} KiB")
print(f"  核对: 2×L×H_kv×D×dtype = {2*num_layers*kv_heads*head_dim*dtype_bytes} bytes —— 与公式不含 B、S 一致")

## 4. MHA / GQA / MQA 三兄弟:一个共享头数的数学

上一节反复强调「用 $H_{kv}$ 而不是 $H$」。两者的关系由注意力机制的三兄弟决定:

| 机制 | 定义 | $H_{kv}$ | 缓存相对 MHA | 论文 |
|---|---|---|---|---|
| **MHA** | 每个 query 头配一个专属 KV 头 | $H_{kv}=H$ | 基准 1× | Vaswani et al. 2017 |
| **GQA** | $H$ 个 query 头分成 $G$ 组,每组共享 1 个 KV 头 | $H_{kv}=H/G$ | $\frac{H}{H_{kv}}$× | [Ainslie et al., EMNLP'23](https://arxiv.org/abs/2305.13245) |
| **MQA** | 所有 query 头共享 1 个 KV 头 | $H_{kv}=1$ | $H$× | [Shazeer, arXiv:1911.02150](https://arxiv.org/abs/1911.02150) |

GQA 论文的核心洞察:decode 阶段 GPU 每步都要**把全部历史 K/V 从 HBM 读一遍**,这是 memory-bound 的;共享 KV 头直接减少**要存、要读的字节数**,而几乎不损失质量——论文原文:「*uptrained GQA achieves quality close to multi-head attention with comparable speed to MQA*」。

> 📄 直觉:相邻 query 头的注意力分布高度相关,共享一组 KV 头(典型 $G=8$)损失可忽略,> 但 KV 账本直接除以 8。这就是 LLaMA-2/3、Qwen、Mistral 全部标配 GQA 的原因。

🎨 **三兄弟对比**。核心一条:缓存缩小倍数 = $H/H_{kv}$。MQA 是 $H$ 倍,MQA 的 $H_{kv}=1$ 是 GQA 的特例,MHA 的 $H_{kv}=H$ 是另一个特例——GQA 在两者之间取了一个质量/速度兼得的中点。

In [ ]:
# -*- coding: utf-8 -*-

def kv_bytes_per_token(num_layers: int, kv_heads: int, head_dim: int,
                       dtype_bytes: int = 2) -> int:
    # 每个 token 的全部层 K/V 字节数: 2(K/V) × L × H_kv × D × dtype_bytes
    return 2 * num_layers * kv_heads * head_dim * dtype_bytes

# GQA 的数学: H 个 query 头分成 G 组, 每组共享 1 个 KV 头 → H_kv = H / G
H_q = 32                                   # H: 总 query 头数(决定表达能力)
G = 8                                      # G: 组数(每组内共享 1 个 KV 头)
H_kv_gqa = H_q // G                        # H_kv = 32/8 = 4 个 KV 头
print(f"GQA: {H_q} 个 query 头 ÷ {G} 组 → H_kv = {H_kv_gqa},缓存缩小 {H_q//H_kv_gqa}×")

# 三种机制并排对比, 看同一模型规模下每 token 成本差多少
print(f"\n{'机制':<14}{'H_kv':>5}{'每token(KiB)':>14}{'相对MHA':>10}")
for label, hk in [("MHA (G=1)", H_q),            # 无共享: H_kv = H
                  ("GQA (G=8)", H_q // G),       # 8 组共享: 主流配置
                  ("MQA (G=H)", 1)]:             # 极端共享: H_kv = 1
    pt = kv_bytes_per_token(32, hk, 128, 2)      # L=32, D=128, bf16
    ratio = H_q / hk                             # 相对 MHA 的缩小比
    print(f"{label:<14}{hk:>5}{pt/1024:>14.1f}{ratio:>10.1f}×")

# 数值核对: 与上一节手工张量公式同源
assert kv_bytes_per_token(3, 4, 8, 2) == 2 * 3 * 4 * 8 * 2    # 断言 L=3,H_kv=4,D=8 的账
print("\n断言通过: 每 token 公式与手工计算一致 ✅")

## 5. 真实规模数字:主流模型的显存账本

把公式代入主流模型的真实配置。所有数字都能用上面的函数复现:

```
LLaMA-3 8B  : L=32, H_kv=8,  D=128  -> 每 token 128 KiB
LLaMA-3 70B : L=80, H_kv=8,  D=128  -> 每 token 320 KiB
GPT-3 175B  : L=96, H_kv=96, D=128  -> 每 token 4.5 MiB (4608 KiB) (MHA 的代价)
```

注意 **32K 上下文 × 高并发** 时,单是 KV 就能吃满一整张 A100。

🚀 **真实规模数字**。两句话读懂:① GQA/MQA 把账本削掉 4~96 倍,这是长上下文能商用化的前提;② 大模型 + 长上下文 + 高并发下,KV 独占半壁江山,内存规划绕不开这个公式。

In [ ]:
# -*- coding: utf-8 -*-

def kv_bytes(num_layers: int, kv_heads: int, head_dim: int, seq_len: int,
             batch: int = 1, dtype_bytes: int = 2) -> int:
    # 总 KV 字节 = 每 token 字节 × 序列长度 × 并发数
    return kv_bytes_per_token(num_layers, kv_heads, head_dim, dtype_bytes) * seq_len * batch

# 主流模型配置: (名称, L, H_kv, D, 标称上下文长度)
models = [
    ("GPT-3 175B (MHA)",  96, 96, 128, 4096),    # MHA: H_kv = H = 96,账本最厚
    ("LLaMA-2 7B (MHA)",  32, 32, 128, 4096),    # 早期 LLaMA 也是 MHA
    ("LLaMA-3 8B (GQA)",  32,  8, 128, 8192),    # GQA-8: 8 个 KV 头
    ("Qwen2.5 7B (GQA)",  28,  4, 128, 32768),   # Qwen 更激进, 4 个 KV 头
    ("Mistral 7B (GQA)",  32,  8, 128, 32768),   # GQA-8
    ("LLaMA-3 70B (GQA)", 80,  8, 128, 32768),   # 大模型 + 长上下文 + GQA
]
print(f"{'模型':<20}{'L':>4}{'H_kv':>5}{'D':>4}{'每token(KiB)':>14}{'单请求32K(GiB)':>16}")
for name, L, Hk, D, ctx in models:
    pt = kv_bytes_per_token(L, Hk, D, 2) / 1024                    # 每 token KiB
    single = kv_bytes(L, Hk, D, 32768, 1, 2) / 2**30               # 单请求 32K 上下文的 GiB
    print(f"{name:<20}{L:>4}{Hk:>5}{D:>4}{pt:>14.1f}{single:>16.2f}")

# 高并发下总账: LLaMA-3 70B, 32 并发, 每请求 32K token
total = kv_bytes(80, 8, 128, 32768, batch=32, dtype_bytes=2) / 2**30
print(f"\nLLaMA-3 70B: 32 并发 × 32K token → KV 共 {total:.2f} GiB")
print(f"  参考: 一张 A100 80G,权重 bf16 已占 140GiB(需多卡),KV 与权重同量级 → 容量由 KV 决定")

## 6. 真实 GPU 分配:理论 vs 实测 ⚖️

公式是纸面账本,现在把它兑现成**真实显存**。`real_ops.kv_bytes_real` 会在 CUDA 上真正构造形状为
$(2,\ B,\ L,\ H_{kv},\ S,\ D)$ 的张量,并通过 `torch.cuda.memory_allocated()` 测出**真实分配的增量字节**,
与理论公式对照。我们同时测 MHA / GQA / MQA 三种 KV 头配置,亲眼看账本随 $H_{kv}$ 缩放。

🚀 **真实分配**。三种配置的 `理论 ≈ 实测`(偏差 <0.01%)——KV 内存公式不是估算,是精确算术。而 $H_{kv}$ 从 32 → 8 → 1,实测显存相应除以 4、再除以 8,与公式的线性关系完全吻合。

In [ ]:
# -*- coding: utf-8 -*-
import sys, os
# 设定环境变量避免 OpenMP 冲突(Windows 上 torch 常见问题)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
# 把 ch02 目录加入模块搜索路径,好导入 real_ops
sys.path.insert(0, r"..\ch02")
from real_ops import kv_bytes_real                       # 真实 CUDA 分配微基准

gi = 2 ** 30                                             # 1 GiB 的字节数
print("三种 KV 头配置在 GPU 上真实分配 (2, B=4, L=32, H_kv, S=2048, D=128) bf16:")
print(f"{'配置':<18}{'理论(GiB)':>12}{'实测(GiB)':>12}{'偏差':>10}")
for name, hk in [("MHA (H_kv=32)", 32),                  # 每 query 头一个 KV 头
                 ("GQA (H_kv=8)", 8),                    # 8 组共享
                 ("MQA (H_kv=1)", 1)]:                   # 全共享 1 头
    r = kv_bytes_real(L=32, kv_heads=hk, head_dim=128, seq=2048, batch=4, dtype="bf16")
    dev = r["device"]                                    # 返回实际设备(如 cuda:0)
    sim = "(仿真)" if r["simulated"] else "(真实CUDA)"    # 无 GPU 时优雅降级
    err = abs(1 - r["alloc_bytes"] / max(r["theory_bytes"], 1)) * 100  # 相对偏差 %
    print(f"{name:<18}{r['theory_bytes']/gi:>12.2f}{r['alloc_bytes']/gi:>12.2f}{err:>9.4f}%  {sim} @ {dev}")

# 关键观察: 理论 ≈ 实测(偏差 <0.01%), 公式分毫不差地兑现为真实显存
print("\n结论: 账本公式与真实 CUDA 分配误差在 0.01% 以内 ✅")

## 7. 复核与精度旋钮:fp8 / e4m3 让账本再砍一半 🔍

上面的「理论 = 实测」成立,前提是该 dtype 每个元素恰好一个标量(bf16/fp16 各 2 字节)。
工程上把账本**再砍一半**的旋钮是 **KV 量化**:用 **fp8(e4m3)每元素 1 字节**存储 KV。
$H_{kv}$ 已经 GQA 到底减不动时,dtype_bytes 从 2→1 让整个账本直接减半——
这正是 vLLM 的 `--kv-cache-dtype fp8`([官方博客](https://vllm.ai/blog/2026-04-22-fp8-kvcache) 实测长上下文 decode 提速),
也是 ch05 第 33 课 `fp8_kv_quant` 的主题。

⚠️ 注意:`kv_bytes_real` 目前只对 bf16/fp16 做真实分配;fp8 没有原生 torch dtype,会退回 fp32,所以对 fp8 我们只看**理论字节**(1 字节/元素)。

✅ 纯理论对照(每元素 2 字节 → 1 字节)。真实 e4m3 KV 的兑现与校准细节放 ch05 专门讲;本课只需记住:精度是继 GQA 之后第二个乘法级的省显存旋钮。

In [ ]:
# -*- coding: utf-8 -*-

def kv_bytes_dtype(L, H_kv, D, S, B, dtype_bytes):
    # 显式收下 dtype_bytes 参数,方便对比不同精度
    return 2 * L * H_kv * D * S * B * dtype_bytes

# LLaMA-3 8B 规模: L=32, GQA-8, D=128; 32K 上下文, 16 并发
L, Hk, D, S, B = 32, 8, 128, 32768, 16
bf16_gb = kv_bytes_dtype(L, Hk, D, S, B, 2) / 2**30        # bf16: 每元素 2 字节
fp8_gb  = kv_bytes_dtype(L, Hk, D, S, B, 1) / 2**30        # fp8 e4m3: 每元素 1 字节
print(f"bf16  KV = {bf16_gb:.2f} GiB")
print(f"fp8   KV = {fp8_gb:.2f} GiB")
print(f"节省 = {100*(1 - fp8_gb/bf16_gb):.0f}%  (dtype_bytes 2 → 1,账本直接减半)")

# 组合拳: GQA 先把 H_kv 从 96 减到 8, fp8 再把字节减半 → 总账除 24
mha = kv_bytes_dtype(96, 96, 128, 32768, 32, 2) / 2**30     # 96 层 MHA-96, bf16
combo = kv_bytes_dtype(96, 8, 128, 32768, 32, 1) / 2**30    # 96 层 GQA-8, fp8
print(f"\n极端对比: 96层 MHA bf16 = {mha:.1f} GiB → 96层 GQA-8 fp8 = {combo:.1f} GiB")
print(f"总缩小 {mha/combo:.0f}× (GQA 除 12 × fp8 除 2)")

## 8. 显存去向可视化:KV 常常是最大的那块 🥧

一张 80GB 的 GPU 上,显存大致分三块:模型权重、KV Cache、激活。激活在推理时很小(约权重的 2%),而 **KV Cache 在长序列 + 高并发下往往超过权重**。下面用饼图看一个典型场景。

🥧 **典型场景**。长上下文 + 高并发下 KV Cache 独占半壁江山甚至反超权重——这正是「显存 = 容量上限」的原因,也为第 10~13 课「怎么省这块内存」埋下伏笔。

In [ ]:
# -*- coding: utf-8 -*-
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"                         # 让 fig.show() 内嵌到 notebook

# 典型场景: LLaMA-3 70B, 80 层 GQA-8, 32K 上下文, 32 并发, bf16
weights_gb = 70e9 * 2 / 1e9                                # 70B 参数 × bf16 2 字节 = 140 GB
kv_gb = kv_bytes(80, 8, 128, 32768, batch=32, dtype_bytes=2) / 1e9   # 全部并发 KV(GB)
act_gb = weights_gb * 0.02                                 # 激活约权重的 2%(推理期很小)
kv_share = kv_gb / (kv_gb + weights_gb + act_gb) * 100     # KV 占显存比例

fig = go.Figure(go.Pie(
    labels=["KV Cache", "模型权重", "激活(估)"],            # 三个组成部分
    values=[kv_gb, weights_gb, act_gb],                     # 对应的 GB 数
    hole=0.45,                                              # 环形饼图
    marker=dict(colors=["#2ecc71", "#3498db", "#f39c12"]),  # 三色区分
    textinfo="label+value+percent"))                        # 同时显示标签/数值/百分比
fig.update_layout(title=f"🥧 推理显存去向: KV Cache 占 {kv_share:.0f}%",
                  template="plotly_white")
fig.show()

## 9. 与 vLLM 工程实现的关系:块预算怎么定

vLLM 启动时会做一次**内存规划**(见 `v1/core/kv_cache_utils.py` 的 `get_kv_cache_config_from_groups`),逻辑如下:

1. 总显存 × `gpu_memory_utilization`(默认 0.9)= 可用预算;
2. 扣除模型权重、激活、CUDA graph 等固定开销 = **KV 预算**;
3. `num_gpu_blocks = KV 预算 ÷ 每块字节数`,其中每块字节 = **每 token 字节 × block_size**(默认 16)。

所以本课的公式**直接决定 vLLM 能缓存多少 token**。我们在代码里把这条路复现一遍。

> ⚠️ 注意权重预算要「放得进一张卡」:LLaMA-3 70B 的 bf16 权重 140 GB 一张 A100(80G)根本装不下,> 所以下面用 **LLaMA-3 8B(权重 16 GB)跑在 A100 80G** 上,贴近真实部署。

🧮 **复现 vLLM 内存规划**。同一个 KV 预算下,$H_{kv}$ 从 32 → 8 → 1,可缓存 token 数线性翻倍。这就是为什么 GQA + fp8 是工业界「塞下更多请求」的两大杀器。

In [ ]:
# -*- coding: utf-8 -*-

# ---- 第 1 步: 总显存预算 ----
gpu_mem_bytes = 80e9                        # 一张 A100/H100 的 80 GB 显存
utilization = 0.9                           # vLLM 默认 gpu_memory_utilization
budget = gpu_mem_bytes * utilization        # 预算 = 总显存 × 利用率
print(f"总显存 {gpu_mem_bytes/1e9:.0f} GB × {utilization} → 预算 {budget/1e9:.1f} GB")

# ---- 第 2 步: 扣除固定开销(权重 + 激活)得到 KV 预算 ----
weights = 8e9 * 2                           # LLaMA-3 8B: 8B 参数 × bf16 2 字节 = 16 GB
activations = weights * 0.02                # 激活约为权重的 2%
kv_budget = budget - weights - activations  # 留给 KV Cache 的字节
print(f"权重 = {weights/1e9:.0f} GB,激活 ≈ {activations/1e9:.1f} GB → KV 预算 = {kv_budget/1e9:.1f} GB")

# ---- 第 3 步: 除以每块字节数得到物理块数量 ----
block_size = 16                             # vLLM 默认块大小: 每块 16 个 token
per_token = kv_bytes_per_token(32, 8, 128, 2)   # LLaMA-3 8B 每 token 字节 = 131,072 (128 KiB)
page_bytes = per_token * block_size         # 每块字节 = 每 token 字节 × 16
num_gpu_blocks = int(kv_budget // page_bytes)   # 物理块数 = KV 预算 ÷ 每块字节
print(f"每块 = {per_token/2**10:.0f} KiB × {block_size} = {page_bytes/2**20:.2f} MiB")
print(f"num_gpu_blocks ≈ {num_gpu_blocks:,}")
print(f"可缓存 token 数 ≈ {num_gpu_blocks*block_size/1e3:.0f}K"
      f"(≈ {num_gpu_blocks*block_size/8192:.1f} 个 8K 请求,或 {num_gpu_blocks*block_size/32768:.1f} 个 32K 请求)")

# ---- 第 4 步: 反推不同 KV 头数对容量的影响(结构上的旋钮) ----
for hk, label in [(32, "MHA (H_kv=32)"), (8, "GQA (H_kv=8)"), (1, "MQA (H_kv=1)")]:
    pt = kv_bytes_per_token(32, hk, 128, 2)     # 该配置下每 token 字节
    blocks = int(kv_budget // (pt * block_size)) # 能分到的物理块数
    print(f"{label:<14} 每token {pt/2**10:6.0f} KiB → 可缓存 {blocks*block_size/1e3:7.0f}K token")

## 10. 🖥️ Streamlit 动态演示:交互式 KV 内存计算器

把公式做成交互 App:拖动**层数 / kv_heads / head_dim / dtype / 序列长度 / 并发数**滑杆,
实时计算每 token / 单请求 / 总 KV 显存,并用柱状图对比 MHA/GQA/MQA、饼图展示显存去向。

### 📜 App 完整源码(`app_08_kv_memory.py`)

▶️ 此 cell 在 streamlit 环境中才真正运行;在 notebook 中仅作展示。

In [ ]:
try:
    import streamlit as st
    _IS_STREAMLIT = bool(st.runtime.exists())
except Exception:
    _IS_STREAMLIT = False

if _IS_STREAMLIT:
    """app_08_kv_memory.py — KV Cache 内存计算器(真实显存分配版)
    运行: python -m streamlit run app_08_kv_memory.py
    """
    import os
    import sys

    _HERE = os.path.dirname(os.path.abspath(__file__))
    if _HERE not in sys.path:
        sys.path.insert(0, _HERE)

    import streamlit as st
    import plotly.graph_objects as go

    from app_common import terminology, real_badge, foot_note
    from real_ops import kv_bytes_real, cuda_available, device_str
    st.set_page_config(page_title="KV Cache 内存计算器", layout="wide")

    st.title("🧮 KV Cache 的内存账本:一个 token 占多少显存?")
    st.markdown(
        "KV Cache 为每个 token 保存**全部层**的 K 与 V,占用公式:\n\n"
        "$$ \\text{KV bytes/token}= 2 \\times L \\times \\text{kv\\_heads} \\times \\text{head\\_dim} \\times \\text{element\\_bytes} $$\n\n"
        "### 🔬 为什么是这个公式?\n"
        "对一个自回归序列,每个 token 在每个头都要预留 **K 与 V 各一份**(因 2);每层都有独立的 KV 投影(因 $L$);"
        "GQA/MQA 里多个 query 头**共享**同一组 KV 头,所以用 $\\text{kv\\_heads}$ 而非全部 $H$ 个头——这正是**省显存的关键旋钮**。\n\n"
        "**本文会真的在 GPU 上按该形状分配一个 KV 张量,量它到底占多少显存**,并与理论公式对照,而不是只敲计算器。"
    )
    st.caption("拖动滑杆,右侧柱状图对比 MHA/GQA/MQA,饼图展示显存去向;KV 显存数字来自真实 torch.cuda 分配。")

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        n_layers = st.slider("🧱 层数 num_layers (L)", 1, 200, 32)
    with c2:
        n_heads = st.slider("🎯 注意力头数 num_heads (H)", 1, 128, 32)
    with c3:
        kv_heads = st.slider("🔑 kv_heads(GQA)", 1, 32, 8)
    with c4:
        head_dim = st.select_slider("📏 head_dim", options=[64, 128, 256], value=128)

    c5, c6, c7, c8 = st.columns(4)
    with c5:
        dtype = st.selectbox("🎛️ 数据类型", ["bf16/fp16 (2 字节)", "fp8 (1 字节)", "fp32 (4 字节)"], index=0)
    with c6:
        seq_len = st.slider("📜 单请求序列长度", 128, 16384, 2048, step=128)
    with c7:
        batch = st.slider("👥 并发请求数", 1, 256, 16)
    with c8:
        run_kv = st.button("🚀 在 GPU 上真实分配 KV 张量", type="primary")

    bytes_per_elem = {"bf16/fp16 (2 字节)": 2, "fp8 (1 字节)": 1, "fp32 (4 字节)": 4}[dtype]
    dtype_key = {"bf16/fp16 (2 字节)": "bf16", "fp8 (1 字节)": "fp16", "fp32 (4 字节)": "fp32"}[dtype]
    per_token = 2 * n_layers * kv_heads * head_dim * bytes_per_elem
    per_req = per_token * seq_len
    total_kv = per_req * batch
    gpu_gb = 16.0
    kv_gb = total_kv / 1e9
    weights_gb = 7.0 * 1e9 * bytes_per_elem / 1e9
    act_gb = weights_gb * 0.02

    # 当用户点按钮且是 CUDA,才真的分配
    kv_real = None
    if run_kv and cuda_available():
        with st.spinner("分配真实 KV 张量并测量显存增量…"):
            kv_real = kv_bytes_real(n_layers, kv_heads, head_dim, seq_len, batch, dtype_key)
    elif run_kv and not cuda_available():
        st.warning("未检测到 CUDA,退化为理论值。")

    def to_gb(x):
        return x / 1e9

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("每 token 占用", f"{per_token/1024:.1f} KiB", help="2 × L × kv_heads × head_dim × bytes")
    m2.metric("单请求 KV", f"{to_gb(per_req):.2f} GB")
    m3.metric("全部并发 KV", f"{kv_gb:.2f} GB",
              delta=f"占显存 {kv_gb/gpu_gb*100:.0f}%" + (" ⚠️" if kv_gb/gpu_gb > 0.5 else ""))
    m4.metric("权重(估)", f"{weights_gb:.1f} GB")

    if kv_real:
        st.info(
            f"📡 **真实分配结果**:在 {kv_real['device']} 上分配形状为 "
            f"(2, {batch}, {n_layers}, {kv_heads}, {seq_len}, {head_dim}) 的 {dtype_key} 张量,"
            f"实测增量 **{kv_real['alloc_bytes']/1e9:.2f} GB**,与理论公式 **{kv_real['theory_bytes']/1e9:.2f} GB** "
            f"误差 {(kv_real['alloc_bytes']-kv_real['theory_bytes'])/max(kv_real['theory_bytes'],1)*100:.2f}%。")
    real = bool(kv_real and not kv_real.get("simulated"))

    fig1 = go.Figure()
    labels = ["MHA (kv=全部头)", "GQA (kv=%d)" % kv_heads, "MQA (kv=1)"]
    vals = [2 * n_layers * n_heads * head_dim * bytes_per_elem * seq_len,
            2 * n_layers * kv_heads * head_dim * bytes_per_elem * seq_len,
            2 * n_layers * 1 * head_dim * bytes_per_elem * seq_len]
    fig1.add_trace(go.Bar(x=labels, y=[to_gb(v) for v in vals],
                          marker_color=["#e74c3c", "#2ecc71", "#3498db"], text=[f"{to_gb(v):.2f} GB" for v in vals],
                          textposition="outside"))
    fig1.update_layout(title="📊 同配置下 MHA / GQA / MQA 的单请求 KV 显存", yaxis_title="GB",
                        template="plotly_white")
    st.plotly_chart(fig1, width="stretch")

    fig2 = go.Figure(go.Pie(
        labels=["KV Cache", "模型权重", "激活(估)"],
        values=[kv_gb, weights_gb, act_gb], hole=0.45,
        marker=dict(colors=["#2ecc71", "#3498db", "#f39c12"]),
        textinfo="label+percent", hovertemplate="%{label}: %{value:.2f} GB (%{percent})"))
    fig2.update_layout(title="🥧 显存去向:KV Cache 常常是最大的那一块", template="plotly_white")
    st.plotly_chart(fig2, width="stretch")
    st.markdown("### 🏷️ 关键结论")
    st.success(f"当前配置每 token 需 **{per_token/1024:.1f} KiB**;{batch} 个请求、各 {seq_len} token 共需 **{kv_gb:.2f} GB**,"
               f"约占 16 GB 显存的 **{kv_gb/gpu_gb*100:.1f}%**。")
    st.markdown(
        "### 💡 为什么 GQA 这么香?\n"
        "MHA 里每个 query 头都带一组专属 KV(**2·H·L·d·seq**),显存随头数线性爆炸;"
        "GQA 让一组 KV 被 4~8 个 query 头共享,**KV 显存直觉上除以 query-per-kv**;"
        "MQA 更是退化为单 KV 组。代价是表达能力略降,但**decode 是 memory-bound**,"
        "省下的带宽直接换吞吐——这就是 LLaMA-2/3 系列普遍用 GQA、Gemma 用 MQA 的原因。\n\n"
        "再把 dtype 换成 fp8/e4m3,元素字节再砍一半——量化与 GQA 是工业界省显存的两大杀器。"
    )
    real_badge(real=True)
    terminology([
        ("KV bytes/token", "一个 token 全部层、K/V 共占的字节数,是显存规划的最小单元。"),
        ("GQA / MQA", "Group-Query Attention 与 Multi-Query Attention:让多个 query 头共享 KV 头以省显存/带宽。"),
        ("e4m3 / e5m2", "FP8 的两种指数长度编码;e4m3 精度更高,vLLM 默认用它做 KV/权重量化。"),
        ("memory access granularity", "对 KV 的读写以固定字节对齐,block 内可能产生 padding(见 09)。"),
        ("vLLM CacheConfig.DEFAULT_BLOCK_SIZE=16", "见 docs/02;块大小决定页粒度,与 half-warp 对齐。"),
    ] + [("出处2", f"真实分配来源:{device_str()}@{dtype_key}")])

    foot_note()

    if __name__ == "__main__":
        try:
            import streamlit.runtime as st_runtime
            if st_runtime.exists():
                raise SystemExit(0)
        except Exception:
            pass
        import os as _os
        import subprocess
        import sys as _sys
        subprocess.run([_sys.executable, "-m", "streamlit", "run", _os.path.abspath(__file__)])
else:
    print("💡 当前不是 streamlit 环境,跳过执行本 App。")
    print("    请把上方源码保存为 app_08_kv_memory.py 后运行:")
    print("    python -m streamlit run app_08_kv_memory.py")


### 🏃 运行方法

1. 使用本目录已生成的 `app_08_kv_memory.py`;
2. 在命令行执行:
```
python -m streamlit run app_08_kv_memory.py
```
3. 浏览器打开 http://localhost:8501 。

🔍 试试:把 kv_heads 从 32 拉到 1(模拟 MQA),总 KV 显存直接除以 32;把 dtype 换成 fp8 再除以 2。

## ✅ 本课小结

- 每 token KV 字节 = 2 × L × H_kv × D × dtype_bytes;总账 = 每 token × S × B,公式是精确算术而非估算
- 用形状 (2, B, L, H_kv, S, D) 的手工张量推演,numel × 字节与公式严格相等,并在 GPU 上实测验证偏差 <0.01%
- GQA 把 H 个 query 头分 G 组共享 KV 头,缓存缩小 H/H_kv 倍;MQA 是 H_kv=1 的特例,MHA 是 H_kv=H 的特例
- 长序列 + 高并发下 KV Cache 常占显存 50%+,与权重平起平坐,是容量规划的第一约束
- vLLM 用「显存预算 × 利用率 − 固定开销」算 KV 预算,再 ÷ 每块字节数得到 num_gpu_blocks;GQA 与 fp8 是两个乘法级的省显存旋钮

## ✍️ 动手练习

1. 把 kv_bytes() 泛化成接收 s_gqa = H/H_kv 的函数,画一条「组数 G vs 单请求 KV」的曲线,验证 G 每翻倍账本减半
2. 用 24GB 显存(RTX 3090)模拟跑 LLaMA-3 70B(GQA-8, fp8):算出能缓存多少 token、几个 32K 请求
3. 查一个你本地模型的 config.json 的 num_key_value_heads,用公式手算它的每 token KV 成本与单请求成本

## 🔗 延伸阅读

- [PagedAttention 论文](https://arxiv.org/abs/2309.06180)
- [GQA 论文 (EMNLP'23)](https://arxiv.org/abs/2305.13245)
- [MQA 论文](https://arxiv.org/abs/1911.02150)
- [vLLM: FP8 KV-Cache 官方博客](https://vllm.ai/blog/2026-04-22-fp8-kvcache)
- [tutorialQ: KV Cache Sizing 手工计算](https://tutorialq.com/ai/dl-infrastructure/explain-kv-cache-sizing)

---
> 🎉 恭喜完成本课!下一课见!